In [7]:
import pandas as pd
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import joblib
from sklearn.metrics import classification_report


base_dir = os.getcwd()
parent_dir = os.path.dirname(base_dir)
data_train_path = os.path.join(parent_dir, "data", "processed", "training.csv")
data_val_path = os.path.join(parent_dir, "data", "processed", "val.csv")
model_path = os.path.join(parent_dir, "models")

In [8]:
df_train = pd.read_csv(data_train_path)
df_val = pd.read_csv(data_val_path)

In [9]:
X_train = df_train['text']
y_train = df_train['label']
X_test = df_val['text']
y_test = df_val['label']

In [10]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_tf = tfidf.fit_transform(X_train)
X_test_tf = tfidf.transform(X_test)

In [11]:
model = LogisticRegression(max_iter=2000, class_weight="balanced")

In [12]:
le = joblib.load(os.path.join(model_path, "label_encoder.pkl"))

d:\DOCUMENT\School\NLP\external\project\twitter_sentiment_classification\.venv1\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [13]:
model.fit(X_train_tf, y_train)
y_pred = model.predict(X_test_tf)

#y_test = le.inverse_transform(y_test)
y_pred = le.inverse_transform(y_pred)

In [14]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1, 5, 10],             # độ regularization
    'solver': ['lbfgs', 'saga'],      # thuật toán tối ưu
    'class_weight': [None, 'balanced']
}

grid = GridSearchCV(
    LogisticRegression(max_iter=20000),
    param_grid,
    cv=3,
    n_jobs=-1,
    scoring='f1_weighted'
)
grid.fit(X_train_tf, y_train)

print("Best params:", grid.best_params_)
model = grid.best_estimator_

Best params: {'C': 0.1, 'class_weight': 'balanced', 'solver': 'lbfgs'}


In [16]:
model.fit(X_train_tf, y_train)
y_pred = model.predict(X_test_tf)

#y_pred = le.inverse_transform(y_pred)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.69      0.82      0.75       265
           1       0.85      0.69      0.76       457
           2       0.72      0.81      0.76       277

    accuracy                           0.76       999
   macro avg       0.75      0.77      0.76       999
weighted avg       0.77      0.76      0.76       999



In [18]:
from sklearn.svm import LinearSVC
model = LinearSVC(C=4)
model.fit(X_train_tf, y_train)
y_pred = model.predict(X_test_tf)

#y_pred = le.inverse_transform(y_pred)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.92      0.92       265
           1       0.92      0.93      0.93       457
           2       0.91      0.90      0.90       277

    accuracy                           0.92       999
   macro avg       0.92      0.92      0.92       999
weighted avg       0.92      0.92      0.92       999

